# Exploring the drive

The INS log of the Oxford RobotCar sample: speed and heading over the 42-minute drive, the turns the model learns to anticipate, and where the 3.6 seconds of camera frames fall. Run it from the repository root's environment (`jupyter lab`, or VS Code); it reads only `oxfordrobocar_sample_small/`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
DATA = os.path.join(ROOT, 'oxfordrobocar_sample_small')

ins = pd.read_csv(os.path.join(DATA, 'gps', 'ins.csv'))
ins['minute'] = (ins['timestamp'] - ins['timestamp'].iloc[0]) / 60e6
ins['speed'] = np.sqrt(ins['velocity_north']**2 + ins['velocity_east']**2)
print(f"{len(ins):,} readings over {ins['minute'].iloc[-1]:.1f} minutes, {len(ins) / (ins['minute'].iloc[-1] * 60):.0f} Hz")
ins[['velocity_north', 'velocity_east', 'velocity_down', 'roll', 'pitch', 'yaw', 'speed']].describe().round(3)

In [ ]:
# Heading change rate over the next second: above 0.1 rad/s is a turn (the model's label)
yaw = np.unwrap(ins['yaw'].to_numpy())
step = 50  # readings per second
turn_rate = np.abs(np.r_[yaw[step:] - yaw[:-step], np.zeros(step)])
ins['turning'] = turn_rate > 0.1
print(f"Turning {ins['turning'].mean():.1%} of the time")

frames = sorted(int(f[:-4]) for f in os.listdir(os.path.join(DATA, 'stereo', 'centre')))
camera = [(t - ins['timestamp'].iloc[0]) / 60e6 for t in (frames[0], frames[-1])]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(ins['minute'], ins['speed'], lw=0.6)
ax1.set_ylabel('speed (m/s)')
ax2.plot(ins['minute'], np.degrees(yaw - yaw[0]), lw=0.6)
ax2.fill_between(ins['minute'], 0, 1, where=ins['turning'], transform=ax2.get_xaxis_transform(), alpha=0.2, label='turning')
ax2.set_ylabel('heading change (deg)')
ax2.set_xlabel('minute of the drive')
for ax in (ax1, ax2):
    ax.axvspan(*camera, color='red', alpha=0.5, label='camera frames')
    ax.axvline(25.3, color='gray', ls='--')
    ax.axvline(33.7, color='gray', ls='--')
ax2.legend(loc='lower left')
ax1.set_title('Training | validation | test (dashed lines)')
plt.tight_layout()